In [13]:
num_entities = 227

In [ ]:
from __future__ import annotations

import json
import os
import time
from pathlib import Path
from urllib.error import HTTPError
from urllib.request import Request, urlopen

import pandas as pd

API_KEY = os.environ.get("GOOGLE_API_KEY") or ''

#csv_path = (Path("../csv") / "merged.csv").resolve()
csv_path = 'merged.csv'
df = pd.read_csv(csv_path)

if "place_id" not in df.columns:
    raise SystemExit("merged.csv is missing a place_id column")

place_ids = df["place_id"].dropna().astype(str)
place_ids = place_ids[place_ids.str.len() > 0].drop_duplicates().tolist()

len(place_ids)

227

In [15]:
FIELD_MASK = ",".join(
    [
        "id",
        "displayName",
        "formattedAddress",
        "location",
        "types",
        "primaryType",
        "primaryTypeDisplayName",
        "businessStatus",
        "rating",
        "userRatingCount",
        "nationalPhoneNumber",
        "internationalPhoneNumber",
        "websiteUri",
        "googleMapsUri",
        "regularOpeningHours",
        "currentOpeningHours",
        "priceLevel",
        "utcOffsetMinutes",
    ]
)


def fetch_place_details(place_id: str) -> dict:
    url = f"https://places.googleapis.com/v1/places/{place_id}"
    req = Request(
        url,
        method="GET",
        headers={
            "X-Goog-Api-Key": API_KEY,
            "X-Goog-FieldMask": FIELD_MASK,
        },
    )
    with urlopen(req, timeout=30) as resp:
        return json.loads(resp.read().decode("utf-8"))


if not API_KEY:
    raise SystemExit("Set GOOGLE_API_KEY in your environment before running.")

n = min(int(num_entities), len(place_ids))
subset = place_ids[:n]

out_rows = []

for i, pid in enumerate(subset, start=1):
    try:
        details = fetch_place_details(pid)
        out_rows.append(
            {
                "place_id": pid,
                "websiteUri": details.get("websiteUri"),
                "nationalPhoneNumber": details.get("nationalPhoneNumber"),
                "internationalPhoneNumber": details.get("internationalPhoneNumber"),
                "googleMapsUri": details.get("googleMapsUri"),
                "place_details_json": json.dumps(details, ensure_ascii=False),
            }
        )
    except HTTPError as e:
        body = e.read().decode("utf-8", errors="replace") if hasattr(e, "read") else ""
        out_rows.append({"place_id": pid, "error": f"{e.code} {e.reason}", "error_body": body})
    time.sleep(0.05)

out = pd.DataFrame(out_rows)

df_enriched = df.merge(out, on="place_id", how="left")

#out_path = (Path("../csv") / "place_details_enriched.csv").resolve()'
out_path = 'place_details_enriched.csv'
df_enriched.to_csv(out_path, index=False)

out_path

'place_details_enriched.csv'